# Tutorial: Long-context memory estimator

Audience:
- `circuit-tracer` users planning attribution runs on long prompts.
- Contributors evaluating why a trace may run out of memory before graph export.

Prerequisites:
- Basic familiarity with attribution graphs and transformer layer/token counts.
- A local checkout or installed copy of `circuit-tracer` from this branch.

Learning goals:
- Estimate dense graph memory without loading a model or transcoders.
- Understand why long contexts create many reconstruction-error nodes.
- Compare dtype and feature-node cap tradeoffs before launching a trace.


## Outline

1. Import the estimator from a local checkout.
2. Estimate a small prompt that should fit comfortably.
3. Estimate a long-context prompt that is likely to fail with dense graph tensors.
4. Compare dtype and feature-node cap choices.
5. Use the CLI equivalent.
6. Exercise: adjust the configuration and inspect the result.


In [ ]:
# Make the notebook work both from the repository root and from demos/.
from __future__ import annotations

from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "demos":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from circuit_tracer.utils.memory_estimation import estimate_graph_memory, format_bytes


## Step 1 - A small prompt estimate

The graph has feature nodes, error nodes, token nodes, and logit nodes. Error nodes scale as `layers * tokens`, so even a modest prompt grows with model depth.


In [ ]:
small = estimate_graph_memory(
    n_tokens=128,
    n_layers=26,
    max_feature_nodes=7500,
    n_logits=10,
    dtype="float16",
    available_memory_gib=24,
)

print(small.to_markdown())


Interpretation: for this configuration, the dense graph tensors are large but usually not the dominant concern on a 24 GiB GPU. The model and transcoder weights still need memory, so this estimate is a lower-level graph preflight rather than a full runtime guarantee.


## Step 2 - A long-context estimate

Now estimate a 6,000-token prompt on a 26-layer model. This mirrors the class of OOM reports where the graph representation itself becomes the bottleneck.


In [ ]:
long_context = estimate_graph_memory(
    n_tokens=6000,
    n_layers=26,
    max_feature_nodes=7500,
    n_logits=10,
    dtype="float16",
    available_memory_gib=80,
)

summary = long_context.to_dict()
print("total nodes:", f"{summary['nodes']['total_nodes']:,}")
print("dense adjacency:", summary["memory"]["dense_adjacency"])
print("estimated peak:", summary["memory"]["estimated_peak"])
print("fits usable memory:", summary["fits_usable_memory"])
print("first recommendation:", summary["recommendations"][0])


Interpretation: even with `float16`, dense adjacency and pruning intermediates can exceed an 80 GiB device budget after a safety margin. This is why shortening the prompt, lowering feature caps, or implementing a sparse/blockwise backend matters.


## Step 3 - Compare dtype choices

Dense graph memory scales linearly with bytes per value. Switching from `float32` to `float16` or `bfloat16` roughly halves the dense floating-point tensor footprint.


In [ ]:
for dtype in ["float32", "bfloat16", "float16"]:
    estimate = estimate_graph_memory(
        n_tokens=2048,
        n_layers=26,
        max_feature_nodes=7500,
        n_logits=10,
        dtype=dtype,
        available_memory_gib=80,
    )
    print(
        f"{dtype:8s}",
        "nodes=", f"{estimate.n_total_nodes:,}",
        "adjacency=", format_bytes(estimate.dense_adjacency_bytes),
        "peak=", format_bytes(estimate.estimated_peak_bytes),
        "fits=", estimate.fits_usable_memory,
    )


## Step 4 - Compare feature-node caps

`max_feature_nodes` affects node count, but for very long prompts the error nodes from `layers * tokens` often dominate. Lowering the feature cap is still useful for exploratory runs.


In [ ]:
for cap in [1000, 2500, 7500]:
    estimate = estimate_graph_memory(
        n_tokens=6000,
        n_layers=26,
        max_feature_nodes=cap,
        n_logits=10,
        dtype="float16",
        available_memory_gib=80,
    )
    print(
        f"cap={cap:>4}",
        "nodes=", f"{estimate.n_total_nodes:,}",
        "peak=", format_bytes(estimate.estimated_peak_bytes),
        "fits=", estimate.fits_usable_memory,
    )


## CLI equivalent

The same estimate is available without Python code:

```bash
circuit-tracer estimate-memory \
  --tokens 6000 \
  --layers 26 \
  --max_feature_nodes 7500 \
  --n_logits 10 \
  --dtype float16 \
  --available_memory_gib 80
```

Use `--format json` when you want machine-readable output for scripts or dashboards.


## Exercise

Try a model with 32 layers and a 3,000-token prompt. Predict whether `float16` fits in a 48 GiB GPU after the default 80% safety margin, then run the next cell.


In [ ]:
exercise = estimate_graph_memory(
    n_tokens=3000,
    n_layers=32,
    max_feature_nodes=5000,
    n_logits=10,
    dtype="float16",
    available_memory_gib=48,
)

print(exercise.to_markdown())


Common pitfall: pruning thresholds reduce the exported graph after dense tensors have already been built. If this estimator says the initial dense graph is too large, increasing pruning thresholds alone will not fix the allocation problem.

Extension: use the estimator across a grid of token counts to choose a safe maximum prompt length for your hardware before running attribution.
